# TTD Dataset Demo

This notebook follows the modality guidance called out in issue #126 and demonstrates:
- Direct SQL
- REST API
- MCP tools
- Agentic access through the orchestrator

In [ ]:
import json
import os
import sqlite3
from pathlib import Path

import pandas as pd
import requests

REQUEST_TIMEOUT = int(os.getenv("TTD_NOTEBOOK_TIMEOUT", "20"))
TTD_NODE_URL = os.getenv("TTD_NODE_URL", "http://ca-ttd-agentic-prod").rstrip("/")
ORCHESTRATOR_URL = os.getenv(
    "ORCHESTRATOR_URL",
    "https://ca-orchestrator-agentic-prod.bluesmoke-f9a71a0e.westus3.azurecontainerapps.io",
).rstrip("/")
JUPYTER_NODE_URL = os.getenv("JUPYTER_NODE_URL", "http://localhost:8002").rstrip("/")
ORCHESTRATOR_SESSION_TOKEN = os.getenv("ORCHESTRATOR_SESSION_TOKEN", "").strip()
TTD_DB_PATH = Path(os.getenv("TTD_DB_PATH", "/app/data/ttd.sqlite3"))

print("TTD_NODE_URL:", TTD_NODE_URL)
print("ORCHESTRATOR_URL:", ORCHESTRATOR_URL)
print("JUPYTER_NODE_URL:", JUPYTER_NODE_URL)
print("ORCHESTRATOR_SESSION_TOKEN set:", bool(ORCHESTRATOR_SESSION_TOKEN))
print("TTD_DB_PATH:", TTD_DB_PATH)
print("pandas version:", pd.__version__)


## Section A: Setup and environment preflight
Validate endpoint reachability before running modality-specific checks.

In [ ]:
checks = [
    ("ttd-health", f"{TTD_NODE_URL}/health"),
    ("ttd-summary", f"{TTD_NODE_URL}/data/summary"),
    ("orchestrator-health", f"{ORCHESTRATOR_URL}/health"),
]

for name, url in checks:
    try:
        resp = requests.get(url, timeout=REQUEST_TIMEOUT)
        print(f"{name:20s} -> {resp.status_code}")
    except Exception as exc:
        print(f"{name:20s} -> ERROR: {exc}")

## Section B: Direct SQL
This section demonstrates SQL-level access in two ways:
1. Direct local SQLite file access (if DB file is available in notebook runtime)
2. SQL submitted through the node `/data/query` route (read-only guard enforced by node)

In [ ]:
# Count-first diagnostics keep this section actionable when a deployed node is healthy but unseeded.
count_sql = "SELECT COUNT(*) AS interactions_count FROM interactions"

direct_sql_query = """
SELECT c.name, t.target_name, i.metric_type, i.metric_value, i.metric_unit
FROM interactions i
JOIN compounds c ON c.id = i.compound_id
JOIN targets t ON t.id = i.target_id
LIMIT 25
"""

print("=== B0) Row count diagnostic ===")
row_count = None
try:
    resp = requests.post(
        f"{TTD_NODE_URL}/data/query",
        json={"sql": count_sql},
        timeout=REQUEST_TIMEOUT,
    )
    payload = resp.json()
    if resp.status_code == 200 and payload.get("rows"):
        row_count = payload["rows"][0].get("interactions_count")
    print("status:", resp.status_code, "interactions_count:", row_count)
except Exception as exc:
    print("count diagnostic error:", exc)

print("\n=== B1) Local SQLite direct SQL ===")
if TTD_DB_PATH.exists():
    with sqlite3.connect(TTD_DB_PATH) as conn:
        rows = conn.execute(direct_sql_query).fetchall()
    print(f"rows={len(rows)}")
    for row in rows[:5]:
        print(row)
else:
    print(f"Local DB not found at {TTD_DB_PATH}; skipping direct-file SQL check.")

print("\n=== B2) SQL via /data/query -> pandas DataFrame ===")
try:
    resp = requests.post(
        f"{TTD_NODE_URL}/data/query",
        json={"sql": direct_sql_query},
        timeout=REQUEST_TIMEOUT,
    )
    print("status:", resp.status_code)
    payload = resp.json()
    rows = payload.get("rows", [])
    df_sql = pd.DataFrame(rows)
    print("row_count:", payload.get("count", len(df_sql)))
    if not df_sql.empty:
        print(df_sql.head(10).to_string(index=False))
    else:
        print("NOTE: Query succeeded but returned zero rows. This usually means the TTD dataset is not seeded in this environment yet.")
        print("Run Section C (/data/summary and /data/tables) to confirm row counts.")
except Exception as exc:
    print("query error:", exc)


## Section C: REST API
Call dataset-specific and generic data routes directly on the TTD node.

In [ ]:
calls = [
    ("summary", "GET", f"{TTD_NODE_URL}/data/summary", None),
    ("tables", "GET", f"{TTD_NODE_URL}/data/tables", None),
    ("ttd-search-egfr", "GET", f"{TTD_NODE_URL}/ttd/search", {"q": "EGFR", "limit": 5}),
    ("ttd-target-p00533", "GET", f"{TTD_NODE_URL}/ttd/target/P00533", None),
]

for label, method, url, params in calls:
    try:
        if method == "GET":
            r = requests.get(url, params=params, timeout=REQUEST_TIMEOUT)
        else:
            r = requests.request(method, url, json=params, timeout=REQUEST_TIMEOUT)
        print(f"\n{label} -> {r.status_code}")
        body = r.json()
        print(json.dumps(body, indent=2)[:1200])
    except Exception as exc:
        print(f"{label} -> ERROR: {exc}")

## Section D: MCP Tools
Discover MCP tools from the TTD node and invoke a tool call.

In [ ]:
mcp_tools = []
try:
    r = requests.get(f"{TTD_NODE_URL}/mcp/tools", timeout=REQUEST_TIMEOUT)
    print("/mcp/tools status:", r.status_code)
    data = r.json()
    mcp_tools = data if isinstance(data, list) else data.get("tools", [])
    print("tool_count:", len(mcp_tools))
    for t in mcp_tools[:10]:
        name = t.get("name") if isinstance(t, dict) else None
        desc = t.get("description") if isinstance(t, dict) else None
        print("-", name, "::", (desc or "")[:120])
except Exception as exc:
    print("mcp tools discovery error:", exc)

In [ ]:
candidate_names = []
for t in mcp_tools:
    if not isinstance(t, dict):
        continue
    nm = (t.get("name") or "").strip()
    if nm:
        candidate_names.append(nm)

search_tool = next((n for n in candidate_names if "search" in n.lower()), None)
target_tool = next((n for n in candidate_names if "target" in n.lower()), None)
tool_name = search_tool or target_tool

print("selected_tool:", tool_name)
if not tool_name:
    print("No usable MCP tool discovered in /mcp/tools output.")
else:
    is_search = "search" in tool_name.lower()
    # Use P00533 so seeded baseline data returns non-empty rows.
    search_args = {"query": "P00533", "limit": 5}
    target_args = {"uniprot_id": "P00533"}

    payload_new = {
        "name": tool_name,
        "arguments": search_args if is_search else target_args,
    }
    payload_legacy = {
        "tool": tool_name,
        "args": search_args if is_search else target_args,
    }
    payload_q_fallback = {
        "name": tool_name,
        "arguments": {"q": "P00533", "limit": 5} if is_search else target_args,
    }

    try:
        resp = requests.post(f"{TTD_NODE_URL}/mcp/call", json=payload_new, timeout=REQUEST_TIMEOUT)
        if resp.status_code >= 400:
            resp = requests.post(f"{TTD_NODE_URL}/mcp/call", json=payload_legacy, timeout=REQUEST_TIMEOUT)
        if resp.status_code >= 400 and is_search:
            resp = requests.post(f"{TTD_NODE_URL}/mcp/call", json=payload_q_fallback, timeout=REQUEST_TIMEOUT)

        print("/mcp/call status:", resp.status_code)
        try:
            body = resp.json()
            rows = body.get("rows", []) if isinstance(body, dict) else []
            print("row_count:", body.get("count") if isinstance(body, dict) else "n/a")
            print(json.dumps(rows[:3], indent=2) if rows else resp.text[:1200])
        except Exception:
            print(resp.text[:1200])
    except Exception as exc:
        print("mcp call error:", exc)


## Section E: Agentic Access (Orchestrator)
Use orchestrator `/chat/stream` so routing and delegation behavior can be observed end-to-end.

In [ ]:
query = "Find high-affinity EGFR interactions from the TTD dataset and summarize top hits."
payload = {"message": query, "agent": "orchestrator"}

print("sending to:", f"{ORCHESTRATOR_URL}/chat/stream")
print("query:", query)

headers = {}
if ORCHESTRATOR_SESSION_TOKEN:
    headers["Authorization"] = f"Bearer {ORCHESTRATOR_SESSION_TOKEN}"

def _local_summary_fallback() -> None:
    print("Using local fallback summary from /ttd/search because orchestrator chat is unavailable in this context.")
    try:
        r = requests.get(
            f"{TTD_NODE_URL}/ttd/search",
            params={"q": "P00533", "limit": 5},
            timeout=REQUEST_TIMEOUT,
        )
        print("fallback status:", r.status_code)
        body = r.json()
        rows = body.get("rows", []) if isinstance(body, dict) else []
        if not rows:
            print("No rows found for fallback summary.")
            return

        print("\nFallback summary (top hits):\n")
        for row in rows[:5]:
            print(
                f"- {row.get('compound')} vs {row.get('target_name')} "
                f"({row.get('uniprot_id')}): {row.get('metric_type')}={row.get('metric_value')} {row.get('metric_unit')}"
            )
    except Exception as exc:
        print("fallback failed:", exc)

try:
    with requests.post(
        f"{ORCHESTRATOR_URL}/chat/stream",
        json=payload,
        headers=headers,
        timeout=REQUEST_TIMEOUT,
        stream=True,
    ) as resp:
        print("status:", resp.status_code)

        if resp.status_code in {401, 403}:
            print("Orchestrator chat requires dashboard auth/session token in this environment.")
            _local_summary_fallback()
        else:
            resp.raise_for_status()
            chunks = []
            for line in resp.iter_lines(decode_unicode=True):
                if not line or not line.startswith("data:"):
                    continue
                raw = line[5:].strip()
                if not raw:
                    continue
                try:
                    evt = json.loads(raw)
                except json.JSONDecodeError:
                    continue
                if evt.get("type") == "token":
                    chunks.append(evt.get("content", ""))
                elif evt.get("type") == "error":
                    print("stream error:", evt)
                    break
                elif evt.get("type") == "done":
                    break

            answer = "".join(chunks).strip()
            print("\nassistant response:\n")
            print(answer if answer else "(no tokens returned)")
except Exception as exc:
    print("agentic call failed:", exc)
    _local_summary_fallback()
